# Why things swing {#sec-swings}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c06-swings.ipynb)

The board of our hospital group has agreed to hire consultants until the waiting list is back at its target. It sounds like the end of the story. It is often the start of a new one: a year later the new posts are still being filled, the list has grown, more posts are approved, and two years after that the group has more consultants than it needs and a hiring freeze. Nobody outside the hospital caused the cycle. It came from the way the decision was made: correcting a stock (staff) through a long delay (recruitment) while forgetting what is already on its way.

This chapter is about that mechanism, **oscillation from stock management**, and how to stop it. The sketch after the setup cell shows the case: the list sets how many consultants are wanted, recruits wait in a pipeline before they start, and the pipeline is the part of the picture that decision makers tend to ignore.

**Who it is for.** Analysts who will model any stock that is managed through a delay: staff, beds, stock on shelves, housing, herds, ships. **Who can skip.** If you can already explain why a supply line weight below 1 makes a stock overshoot, and relate the period of a cycle to the delays in the loop, read section 5 (the running case) and the scorecard, then move on. **Prerequisites.** Chapter 3, Loops that grow, settle and tip (balancing loops and adjustment times); chapter 4, Delays, pipelines and memory (material delays); and chapter 5, Decisions inside the model (decision rules that anchor on a target). **Time.** About three hours with the exercises.

The waiting-list data are synthetic, shaped like published Irish waiting-list counts; see chapter 9, Data, calibration and uncertainty.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package,
# which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 6
rng = np.random.default_rng(SEED)

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])

# Okabe-Ito colours with a different line style each, readable in greyscale
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
             "#E69F00", "#56B4E9", "#000000", "#F0E442"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1)),
          (0, (1, 3)), (0, (8, 3))]
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__, "| data:", wl.shape[0], "months,",
      wl.month.min().strftime("%b %Y"), "to", wl.month.max().strftime("%b %Y"))

In [ ]:
#| fig-alt: Stock-and-flow sketch. Referrals fill the Waiting list and Treatments drain it. Below, Recruiting fills Consultants in pipeline, Starting moves them into Consultants, and Leaving drains Consultants. Dashed information arrows run from the list to Recruiting, from Consultants to Treatments, and from the pipeline back to Recruiting, marked often ignored.
from matplotlib.patches import Circle, FancyBboxPatch


def draw_sfd(ax, stocks, flows, links):
    """A small stock-and-flow diagram.

    stocks: {name: (x, y)}. flows: [(label, (x0, y0), (x1, y1))], drawn as
    pipes with a valve; an end outside every stock gets a cloud.
    links: [(start, end, label, curvature, label position)], dashed arrows.
    """
    w, h = 1.7, 0.8
    for name, (x, y) in stocks.items():
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                                    boxstyle="round,pad=0.02", fc="white",
                                    ec="black", lw=1.5))
        ax.text(x, y, name, ha="center", va="center", fontsize=9)

    def inside(p):
        return any(abs(p[0] - x) <= w / 2 + 1e-9 and abs(p[1] - y) <= h / 2
                   for x, y in stocks.values())

    for label, a, b in flows:
        for p in (a, b):
            if not inside(p):
                ax.text(*p, "☁", ha="center", va="center",
                        fontsize=18, color="grey")
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="-|>", lw=3, color="black",
                                    shrinkA=8, shrinkB=2))
        mx, my = (a[0] + b[0]) / 2, (a[1] + b[1]) / 2
        ax.add_patch(Circle((mx, my), 0.12, fc="white", ec="black",
                            lw=1.2, zorder=3))
        ax.text(mx, my - 0.3, label, ha="center", va="top", fontsize=8)
    for a, b, label, rad, where in links:
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="->", lw=1, ls="--",
                                    color="#0072B2", shrinkA=4, shrinkB=4,
                                    connectionstyle=f"arc3,rad={rad}"))
        ax.text(*where, label, ha="center", fontsize=8, color="#0072B2")
    ax.set_aspect("equal")
    ax.axis("off")


fig, ax = plt.subplots(figsize=(8, 3.8))
draw_sfd(ax,
         stocks={"Waiting list": (4, 3), "Consultants\nin pipeline": (3, 0),
                 "Consultants": (6.5, 0)},
         flows=[("Referrals", (1.5, 3), (3.15, 3)),
                ("Treatments", (4.85, 3), (7, 3)),
                ("Recruiting", (0.4, 0), (2.15, 0)),
                ("Starting", (3.85, 0), (5.65, 0)),
                ("Leaving", (7.35, 0), (9, 0))],
         links=[((3.6, 2.6), (1.3, 0.1), "list above target", 0.3, (1.9, 1.9)),
                ((6.5, 0.4), (5.9, 2.9), "capacity", -0.2, (6.9, 1.5)),
                ((2.6, -0.4), (1.2, -0.1), "often ignored", -0.6,
                 (1.2, -1.0))])
ax.set_xlim(-0.2, 9.6)
ax.set_ylim(-1.3, 3.7)
ax.set_title("Hiring consultants to bring a waiting list down")
plt.show()

Figure: The chapter's case. Thick arrows are flows (patients above, consultants below); dashed arrows carry information. The list above its target asks for more consultants; consultants set capacity, and so treatments. The link from the pipeline back to the hiring decision is the one that is often missing.

**Learning outcomes.** By the end of the chapter you can:

1. **build** a stock management rule with a supply line and **predict** whether it will overshoot (Try it 1 and 3);
2. **diagnose** a swing in data as endogenous (set by delays in the system) or exogenous (set by the calendar) using its period (Try it 3);
3. **build** a workforce from hiring, leaving and a desired level worked out from the workload (Try it 4);
4. **justify** counting the recruitment pipeline in a hiring rule, and **quantify** what the fix costs, on the running case (Try it 5 and the mini-project);
5. **explain** commodity, housing and labour-market cycles with the same structure and **compare** pricing rules (Try it 6).

## 1. Managing a stock you cannot fill at once

Whenever someone keeps a stock near a target and new units take time to arrive, the decision has the same three parts: replace what is being used, close the gap between the desired and the actual stock, and keep enough on order to cover the delay. The units already ordered but not yet received form the **supply line** (also called the pipeline). The **Stock Management** molecule (`stockmgmt`), after Hines, *Molecules of Structure* [@hines_molecules], writes the order rule as

$$
\text{Orders} = \max\!\Big(0,\ D + \frac{S^{*} - S}{T_{\text{adj}}} + w\,\frac{SL^{*} - SL}{T_{\text{adj}}}\Big), \qquad SL^{*} = D \times \tau ,
$$

with demand $D$, inventory $S$ and its target $S^{*}$, supply line $SL$ and its target $SL^{*}$, the **acquisition delay** $\tau$ (a delay time) and one adjustment time $T_{\text{adj}}$ for both gaps. The **supply line weight** $w$, a fraction between 0 and 1, says how much of the pipeline gap the decision maker counts. At $w = 1$ the rule sees everything on order; at $w = 0$ it sees only the shelf. Our version uses a single adjustment time for both gaps (Hines lets them differ), and its `Sales` are limited by a minimum shipping time so the inventory cannot go negative.

The molecule starts with 60 units on the shelf against a target of 100, 10 units a week of demand and a 6-week acquisition delay. Its two presets differ only in the supply line weight.

In [ ]:
#| fig-alt: Two panels over 60 weeks. Left, inventory rises smoothly to 100 when the supply line is counted, and overshoots to about 125 and swings when it is ignored. Right, orders for the same two runs.
sm = lb.molecule("stockmgmt")
print(sm.equations())

runs = {label: sm.run(preset=label)
        for label in ("Stable (sees pipeline)", "Oscillation (ignores it)")}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
for label, d in runs.items():
    d["Inventory"].plot(ax=a1, label=label)
    d["Orders"].plot(ax=a2, label=label)
a1.axhline(100, color="grey", lw=1, ls=":")
a1.set(title="Inventory", xlabel="time (week)", ylabel="inventory (units)")
a2.set(title="Orders", xlabel="time (week)", ylabel="orders (units/week)")
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()

for label, d in runs.items():
    inv, orders = d["Inventory"], d["Orders"]
    print(f"{label:<25} inventory peak {inv.max():6.1f} in week "
          f"{inv.idxmax():4.1f}, orders {orders.min():4.1f} to "
          f"{orders.max():4.1f} units/week")

Figure: Stock Management with the pipeline counted (supply line weight 1) and ignored (weight 0). The dotted line is the target of 100 units.

Both rules start by ordering 20 units a week, twice demand, because both see the same gap of 40 on the shelf. The difference comes in week 2 or 3. The rule that counts the pipeline sees that the gap is already covered by goods on the way and cuts back: inventory rises to 100 and stays there. The rule that ignores the pipeline keeps ordering while the shelf is still low, so by the time the goods arrive it has ordered too many. Inventory overshoots to 110.2 units in week 16.8, orders fall to 7.5 units a week (below demand) to work the excess off, and the stock undershoots in turn. That is the whole mechanism of this chapter: **a balancing loop with a delay that the decision ignores overshoots, and an overshoot in one direction sets up the next in the other.**

**Try it 1.**

- *Check.* What is the desired supply line at the default settings, and why that number?
- *Apply.* Run the molecule for 120 weeks with a supply line weight of 0.5. How big is the first overshoot, and how many peaks follow?
- *Extend.* The *Long lead time* preset doubles the delay to 12 weeks and counts only 30% of the pipeline. How far does inventory overshoot, and what does counting the pipeline in full do?

<details><summary>Worked answer</summary>

The desired supply line is 60 units: 10 units a week of demand times a 6-week delay. That is how much must be on order at any moment for deliveries to match demand (Little's law, from chapter 4). With a weight of 0.5 inventory peaks at 101.7 in week 21.7 and then at 100.0 in weeks 65 and 108: half attention removes almost all the overshoot. With the long lead time, inventory peaks at 105.2 in week 25.9; counting the pipeline in full brings the peak back to 100.0, no overshoot at all, even with the 12-week delay. A long delay alone does not cause a swing; a long delay that the rule does not see does.
</details>

In [ ]:
def peaks(series):
    """(time, value) of every local peak of a series."""
    v, t = series.values, series.index
    return [(float(t[i]), float(v[i])) for i in range(1, len(v) - 1)
            if v[i - 1] < v[i] >= v[i + 1]]


# Check: the desired supply line at the default settings
print("desired supply line:", sm.run(T=0)["Desired supply line"].iloc[0],
      "units")
# Apply: a supply line weight of 0.5
half = sm.run(T=120, supply_line_weight=0.5)["Inventory"]
print("weight 0.5: peaks", [(t, round(v, 1)) for t, v in peaks(half)])
# Extend: the Long lead time preset
long = sm.run(T=120, preset="Long lead time")["Inventory"]
pk = peaks(long)
print("Long lead time: first peaks", [(t, round(v, 1)) for t, v in pk[:3]],
      "| low", round(long.min(), 1))
print("Long lead time, counted in full:",
      round(sm.run(T=120, preset="Long lead time",
                   supply_line_weight=1)["Inventory"].max(), 1))

## 2. When the pipeline is hard to see

Counting the pipeline needs data on it. A shop knows its open purchase orders, but a hospital rarely has one list of posts approved, advertised, offered and accepted, with the expected start dates. The **Low-Visibility Pipeline Correction** molecule (`lowVisPipeline`), after Hines, *Molecules of Structure* [@hines_molecules], models a manager who cannot see inside the supplier and works from what is known: the orders placed but not yet received ($ONR$). The delivery delay is inferred from it, and the required pipeline follows:

$$
\hat\tau = \frac{ONR}{\text{Receiving}}, \qquad
\text{Ordering} = \max\!\Big(0,\ \text{Shipping} + \frac{S^{*} - S}{T_{S}} + a\,\frac{\text{Shipping}\times\hat\tau - ONR}{T_{SL}}\Big).
$$

The **awareness of pipeline** $a$ (0 to 1) plays the part of the supply line weight. The supplier itself is a two-stage delay with a total delay time of 6 weeks, hidden from the manager. Our version caps the inferred delay at 30 weeks so it cannot blow up when receipts stop; that cap is ours, not Hines'.

In [ ]:
#| fig-alt: Inventory over 120 weeks for three levels of awareness of the pipeline. With full awareness inventory rises to 100 without overshoot; with none it overshoots to about 117 and keeps swinging. A second panel shows the delivery delay the manager infers, near 6 weeks.
lv = lb.molecule("lowVisPipeline")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
for aware in (1, 0.5, 0):
    d = lv.run(T=120, awareness_of_pipeline=aware)
    d["Inventory"].plot(ax=a1, label=f"awareness {aware}")
    d["Calculated delivery delay"].plot(ax=a2, label=f"awareness {aware}")
    pk = peaks(d["Inventory"])
    print(f"awareness {aware:<3}: inventory peak {d['Inventory'].max():5.1f},"
          f" orders {d['Ordering'].min():4.1f} to {d['Ordering'].max():4.1f}"
          f" units/week, {len(pk)} peaks in 120 weeks")
a1.axhline(100, color="grey", lw=1, ls=":")
a1.set(title="Inventory", xlabel="time (week)", ylabel="inventory (units)")
a2.set(title="Delivery delay the manager infers", xlabel="time (week)",
       ylabel="delay (week)")
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()

Figure: Low-Visibility Pipeline Correction for three levels of awareness. Left, inventory against its target of 100 units; right, the delivery delay the manager infers from orders not received.

Full awareness gives a smooth rise to 100 with no peak at all. Half awareness overshoots slightly, to 102.9. With none, inventory overshoots to 117.2 and orders swing between 5.7 and 20 units a week when only 10 are shipped. The inferred delay is a useful by-product: it moves above and below the true 6 weeks, because receipts lag behind orders. Even a manager who tracks the pipeline sees the delay only through a noisy lens, which is one reason a full weight is hard to achieve in practice.

**Try it 2.**

- *Check.* In the run with no awareness, at which weeks does inventory peak? What is the time between peaks?
- *Apply.* Keep awareness at 0 and set the supplier delay to 3, 6 and 12 weeks. How do the first peak and the period change?
- *Extend.* Is the period a multiple of the supplier delay? Explain what else sets it.

<details><summary>Worked answer</summary>

Inventory peaks in weeks 16.4, 45.2, 74.1 and 103.0, so one cycle takes about 28.9 weeks. With delays of 3, 6 and 12 weeks the first peak is 106.5, 117.2 and 130.5, and the period is about 21.4, 28.9 and 42.4 weeks. The period grows with the delay but not in proportion: doubling the delay from 6 to 12 weeks makes the cycle about 47% longer, not 100%. The period depends on all the delays in the loop, including the 4 weeks the manager takes to correct inventory, which does not change here. A rule of thumb that matches many such models: the period is a few times the sum of the delays and adjustment times in the loop.
</details>

In [ ]:
blind = lv.run(T=120, awareness_of_pipeline=0)["Inventory"]
pk = peaks(blind)
print("peaks in weeks", [round(t, 1) for t, _ in pk])
print("mean gap between peaks:",
      round(np.diff([t for t, _ in pk]).mean(), 1), "weeks")
for delay in (3, 6, 12):
    inv = lv.run(T=120, awareness_of_pipeline=0,
                 supplier_delay=delay)["Inventory"]
    gaps = np.diff([t for t, _ in peaks(inv)])
    print(f"supplier delay {delay:>2} weeks: peak {inv.max():5.1f}, "
          f"period about {gaps.mean():4.1f} weeks")

## 3. Anatomy of a swing: amplitude and period

Two numbers describe a swing: its **amplitude** (how far it overshoots) and its **period** (how long one full cycle lasts). Both come from the structure, so we can map them. `run_batch` runs Stock Management for every combination of 7 acquisition delays and 3 supply line weights in one call, and a helper measures the period as the mean time between peaks.

In [ ]:
#| fig-alt: Two panels. Left, the first overshoot of inventory above its target of 100 against the acquisition delay, for supply line weights 0, 0.5 and 1; it grows with the delay when the pipeline is ignored and stays near zero when it is counted. Right, the period of the swing against the delay; it lengthens with the delay.
delays = [2, 3, 4, 6, 8, 10, 12]
weights = [0, 0.5, 1]
grid = pd.DataFrame([{"acquisition_delay": a, "supply_line_weight": w}
                     for w in weights for a in delays])
out = sm.run_batch(grid, T=200)
inv = out[out.variable == "Inventory"].pivot(index="time", columns="run",
                                              values="value")


def period(series):
    """Mean time between peaks, or NaN when there are fewer than two."""
    t = [p[0] for p in peaks(series)]
    return np.diff(t).mean() if len(t) > 1 else np.nan


grid["overshoot"] = inv.max() - 100
grid["period"] = [period(inv[r]) for r in grid.index]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
for w, g in grid.groupby("supply_line_weight"):
    a1.plot(g.acquisition_delay, g.overshoot, marker="o",
            label=f"supply line weight {w}")
    a2.plot(g.acquisition_delay, g.period, marker="o",
            label=f"supply line weight {w}")
a1.set(title="How far inventory overshoots", xlabel="acquisition delay (week)",
       ylabel="peak above target (units)")
a2.set(title="How long one swing lasts", xlabel="acquisition delay (week)",
       ylabel="period (week)")
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(grid.pivot(index="acquisition_delay", columns="supply_line_weight",
                 values=["overshoot", "period"]).round(1))

Figure: Overshoot and period of the inventory swing in Stock Management, for acquisition delays from 2 to 12 weeks and three supply line weights. A missing point means there is no cycle to measure.

With the pipeline ignored, the overshoot grows from 1.9 units at a 2-week delay to 15.9 units at 12 weeks, and the period from 24.0 to 45.3 weeks. Half weight keeps the overshoot under 2 units at every delay; the swing that is left is so damped that its period is longer still (36.8 to 62.2 weeks). With full weight there is no cycle at all. The lesson for the analyst: **the period of an endogenous swing is set by the delays in the loop, and so it changes when they change.**

That gives a test for data. A swing whose period stays fixed at 12 months, peaking in the same months every year, is driven from outside, by the calendar. A swing whose period is several times the delays in the system, and drifts as they change, is probably the system's own. The waiting-list data have only the stock, so we look at its month-to-month changes before the 2020 shock and correlate them with themselves at lags of 1 to 24 months (the **autocorrelation**).

In [ ]:
#| fig-alt: "Bar chart of the autocorrelation of month-to-month changes in the waiting list before March 2020, for lags of 1 to 24 months. It is high at lags 1, 11 and 12, and negative around 6 months: a yearly pattern."
def autocorr(x, lags):
    """Correlation of a series with itself shifted by each lag."""
    x = np.asarray(x, dtype=float) - np.mean(x)
    return np.array([np.corrcoef(x[:-k], x[k:])[0, 1] for k in lags])


pre = wl[wl.month < "2020-03-01"]
change = pre.total.diff().dropna()            # patients per month
lags = np.arange(1, 25)
acf = autocorr(change, lags)
fig, ax = plt.subplots()
ax.bar(lags, acf, color="#0072B2")
ax.axhline(0, color="black", lw=0.8)
ax.set(title="Autocorrelation of monthly changes in the list, 2014 to 2020",
       xlabel="lag (month)", ylabel="correlation")
plt.show()
print("largest correlations at lags", lags[np.argsort(acf)[::-1][:3]],
      np.round(np.sort(acf)[::-1][:3], 2))
by_month = pre.assign(change=pre.total.diff(),
                      calendar_month=pre.month.dt.month)
print("mean change by calendar month (patients):")
print(by_month.groupby("calendar_month").change.mean().round(0)
      .to_frame().T.to_string(index=False))

Figure: Autocorrelation of monthly changes in the waiting list, January 2014 to February 2020. Synthetic data shaped like published Irish counts.

The changes are most alike 11 months apart (correlation 0.46), 1 month apart (0.45) and 12 months apart (0.35), and most unalike around 6 months apart. The list grows by about 300 patients a month in May and June and falls by 74 in December: a yearly pattern in referrals, not a cycle made by the hospital. If the group's hiring rule had been making the list swing, we would expect a peak in the autocorrelation at a lag set by recruitment, several years long, and ten years of data would show it only a couple of times. That is the honest limit of the test: a short series can reject a seasonal story more easily than it can prove an endogenous one.

**Try it 3.**

- *Check.* From the table above, what is the period of the swing with a 6-week delay and no weight on the pipeline?
- *Apply.* Run that case for 200 weeks and apply `autocorr` to the weekly changes in inventory. At which lag between 6 and 40 weeks is the correlation highest?
- *Extend.* Why would the same test on a model with a fixed 12-month seasonal input find a peak at 12 whatever the delays?

<details><summary>Worked answer</summary>

The period is 33.4 weeks. The autocorrelation of the model's weekly changes peaks at a lag of 33 weeks, the same number found from the peaks: the two methods agree. A seasonal input repeats every 12 months by construction, so the stock's changes repeat at that lag whatever the internal delays; the delays change only how big the response is and how late it comes. The period of an exogenous swing belongs to the input, the period of an endogenous one to the structure.
</details>

In [ ]:
# Check: the period for an acquisition delay of 6 weeks, ignoring the pipeline
print("period, delay 6, weight 0:",
      round(grid.query("acquisition_delay == 6 and "
                       "supply_line_weight == 0").period.iloc[0], 1), "weeks")
# Apply: the model's own swing has no fixed calendar; check its autocorrelation
blind = sm.run(T=200, supply_line_weight=0, acquisition_delay=6)
weekly = blind["Inventory"].iloc[::10]          # one value a week
acf_model = autocorr(weekly.diff().dropna(), np.arange(1, 41))
print("model: autocorrelation peaks at lag",
      int(np.arange(1, 41)[np.argmax(acf_model[5:]) + 5]), "weeks")

## 4. People are a stock too

Staff are hired and leave, and the workforce is what those flows leave behind. Three molecules give the pieces we need for the running case.

The **Workforce** molecule (`workforce`), after Hines, *Molecules of Structure* [@hines_molecules], closes the gap between the desired and the actual workforce $W$ over a time to hire or fire $T_h$:

$$
\text{Hiring} = \max\!\Big(0, \frac{W^{*} - W}{T_h}\Big), \qquad \text{Firing} = \max\!\Big(0, \frac{W - W^{*}}{T_h}\Big).
$$

The **Desired Workforce from Workflow** molecule (`desiredWorkforce`), after Hines, *Molecules of Structure* [@hines_molecules], works out where $W^{*}$ comes from: the rate at which work must be done divided by what one person does, $W^{*} = R^{*} / p$. It is the Producing molecule of chapter 5 run backwards. The **Building Inventory by Doing Work** molecule (`buildingInventory`), after Hines, *Molecules of Structure* [@hines_molecules], runs it forwards: people at work fill a stock at the rate $W \times p$. Our versions use first-order adjustment in both directions; Hines notes that real organisations often shrink only by attrition, which the exercise below adds.

In [ ]:
#| fig-alt: Three panels. Left, a workforce closing the gap from 40 to 80 people over 20 years. Middle, hiring when 30 people are needed and 10 are on hand, starting at 5 a week and fading. Right, inventory built by 10 workers at 2 units each a month, rising in a straight line.
wf = lb.molecule("workforce")
dw = lb.molecule("desiredWorkforce")
bi = lb.molecule("buildingInventory")

ramp = wf.run(T=20, desired_workforce=80)
team = dw.run(T=30)
build = bi.run(T=24)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
ramp["Workforce"].plot(ax=axes[0])
axes[0].axhline(80, color="grey", lw=1, ls=":")
axes[0].set(title="Workforce: target 80", xlabel="time (year)",
            ylabel="workforce (people)")
team["Hiring"].plot(ax=axes[1])
axes[1].set(title="Desired workforce: hiring", xlabel="time (week)",
            ylabel="hiring (people/week)")
build["Inventory"].plot(ax=axes[2])
axes[2].set(title="Building inventory", xlabel="time (month)",
            ylabel="inventory (units)")
plt.tight_layout()
plt.show()

print("workforce in years 0, 4, 8, 12:",
      ramp.loc[[0.0, 4.0, 8.0, 12.0], "Workforce"].round(1).tolist())
print(f"desired people {team['Desired people'].iloc[0]:.0f}; hiring starts "
      f"at {team['Hiring'].iloc[0]:.1f} people/week, "
      f"{team.loc[12.0, 'Hiring']:.2f} by week 12")
print(f"inventory after 24 months: {build['Inventory'].iloc[-1]:.0f} units "
      f"(starts at {build['Inventory'].iloc[0]:.0f})")

Figure: Three people molecules. Left, Workforce closing a gap from 40 to 80 people (time to hire or fire 4 years). Middle, Desired Workforce: hiring when 120 tasks a week at 4 tasks a person call for 30 people and 10 are on hand. Right, Building Inventory: 10 workers producing 2 units each a month with nothing shipped.

The workforce reaches 65.5 after one hiring time, 74.7 after two and 78.1 after three: the gap shrinks exponentially, as in Close Gap. In the middle panel 30 people are wanted, so hiring jumps at once to (30 minus 10) / 4 = 5.0 people a week and has faded to 0.24 by week 12. Most of the recruiting load lands at the start. The right panel is the plainest of the three, and the one the running case needs most: 10 workers at 2 units each a month add 20 units a month, so inventory climbs in a straight line from 100 to 580 units in 24 months. Read "workers" as consultants, "productivity" as patients seen per consultant and "inventory" as patients treated, and you have the capacity side of our hospital.

**Try it 4.**

- *Check.* In Building Inventory, what happens when shipments equal production (the *Shipments = production* preset)?
- *Apply.* In Desired Workforce, productivity rises from 4 to 6 tasks per person per week. How many people are needed, and how close is the team after 12 weeks?
- *Extend.* Hospitals rarely fire consultants. Change Workforce so that people leave at 5% a year and the firm only stops replacing them. How long does it take to fall from 40 to within one person of 20, compared with firing?

<details><summary>Worked answer</summary>

Inventory stays at 100 units: inflow and outflow are both 20 units a month, so the stock is in equilibrium. With productivity 6 only 20 people are needed (120 / 6); the team of 10 reaches 19.5 by week 12. Without firing, the workforce takes until year 15.8 to come within one person of 20, against 11.9 years with firing. Shrinking by attrition is slower than growing by hiring, and that asymmetry is exactly what makes over-hiring expensive in the next section: excess consultants stay for years.
</details>

In [ ]:
# Check: steady state of Building inventory when shipments equal production
eq = bi.run(T=24, preset="Shipments = production")["Inventory"]
print("shipments = production: inventory", eq.iloc[0], "to",
      round(eq.iloc[-1], 1))
# Apply: productivity rises from 4 to 6 tasks per person per week
fast = dw.run(T=30, productivity=6)
print(f"productivity 6: desired people {fast['Desired people'].iloc[0]:.0f},"
      f" workforce at week 12 {fast.loc[12.0, 'Workforce']:.1f}")
# Extend: no firing; people leave at 5% a year and are replaced as needed
shrink = wf.with_equation("Firing", "Workforce * 0.05").with_equation(
    "Hiring", "MAX(0, Workforce_adjustment + Workforce * 0.05)")
for label, model in (("firing", wf), ("attrition only", shrink)):
    w = model.run(T=40, desired_workforce=20)["Workforce"]
    print(f"{label:<15}: within one person of 20 by year "
          f"{w.index[w <= 21][0]:.1f}")

## 5. The running case: hiring to a moving target

Now we put the pieces together for the hospital. The list from chapter 1 keeps its flows: `Referrals` of 4,000 patients a month, rising by 10% (400 patients a month) in month 12, and `Treatments = MIN(Capacity, Waiting_list / Minimum_treatment_time)` with a minimum treatment time of 0.5 month. Capacity is now made by people, as in Building Inventory: 79 consultants (whole-time equivalents, WTE), each with a team that sees 50 patients a month, give 3,950 patients a month. We treat the nurses as part of each consultant's team for now and split them out later in this section.

The hiring rule is Stock Management with consultants as the stock. How many consultants are needed comes from Desired Workforce, with one twist: the work to be done depends on the list itself,

$$
R^{*} = \text{Referrals} + \frac{\text{Waiting list} - \text{Target list}}{T_{\text{list}}}, \qquad W^{*} = \frac{R^{*}}{p},
$$

so the target for consultants moves whenever the list moves. That is the **moving target**: hiring changes the list, and the list changes how many consultants are wanted. Recruits wait in a pipeline for a **hiring delay** of 12 months (advertising, interviews, notice periods, registration), and consultants leave after an average tenure of 120 months. The recruiting rule replaces leavers, closes the staff gap over a staff adjustment time of 3 months and, with weight `Pipeline_weight`, closes the gap in the pipeline. The pipeline target uses the `Consultants believed delay`, which equals the true delay unless we say otherwise. The function below builds the model for any list of staff groups, so we can add nurses later without writing it again.

In [ ]:
def hiring_spec(groups, rise=400):
    """The waiting-list model with one hiring pipeline per staff group.

    groups: list of (name, starting WTE, productivity in patients per WTE
    per month, hiring delay in months). Referrals rise by `rise` patients
    a month in month 12. Capacity is set by the scarcest group.
    """
    stocks = [{"id": "wl", "name": "Waiting list", "init": "32000",
               "units": "patients", "nonNegative": True}]
    flows = [
        {"id": "ref", "name": "Referrals", "to": "wl",
         "eq": "4000 + STEP(Referral_rise, 12)", "units": "patients/month"},
        {"id": "treat", "name": "Treatments", "from": "wl",
         "eq": "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
         "units": "patients/month"}]
    vars_ = [
        {"id": "rise", "name": "Referral rise", "eq": str(rise),
         "units": "patients/month"},
        {"id": "mtt", "name": "Minimum treatment time", "eq": "0.5",
         "units": "month"},
        {"id": "tgt", "name": "Target list", "eq": "32000",
         "units": "patients"},
        {"id": "tred", "name": "Time to reduce list", "eq": "6",
         "units": "month"},
        {"id": "dtr", "name": "Desired treatments", "units": "patients/month",
         "eq": "MAX(0, Referrals + (Waiting_list - Target_list)"
               " / Time_to_reduce_list)"},
        {"id": "sat", "name": "Staff adjustment time", "eq": "3",
         "units": "month"},
        {"id": "w", "name": "Pipeline weight", "eq": "0",
         "units": "dimensionless"},
        {"id": "ten", "name": "Average tenure", "eq": "120",
         "units": "month"}]
    caps = []
    for i, (name, staff0, pdy, delay) in enumerate(groups):
        k, g = name.replace(" ", "_"), f"g{i}"
        stocks += [
            {"id": g + "p", "name": f"{name} in pipeline", "units": "WTE",
             "init": f"{k}_leaving * {k}_hiring_delay", "nonNegative": True},
            {"id": g, "name": name, "init": str(staff0), "units": "WTE",
             "nonNegative": True}]
        flows += [
            {"id": g + "r", "name": f"{name} recruiting", "to": g + "p",
             "units": "WTE/month",
             "eq": f"MAX(0, {k}_leaving"
                   f" + ({k}_needed - {k}) / Staff_adjustment_time"
                   f" + Pipeline_weight * ({k}_leaving * {k}_believed_delay"
                   f" - {k}_in_pipeline) / Staff_adjustment_time)"},
            {"id": g + "s", "name": f"{name} starting", "from": g + "p",
             "to": g, "units": "WTE/month",
             "eq": f"{k}_in_pipeline / {k}_hiring_delay"},
            {"id": g + "l", "name": f"{name} leaving", "from": g,
             "units": "WTE/month", "eq": f"{k} / Average_tenure"}]
        vars_ += [
            {"id": g + "pd", "name": f"{name} productivity", "eq": str(pdy),
             "units": "patients/WTE/month"},
            {"id": g + "hd", "name": f"{name} hiring delay", "eq": str(delay),
             "units": "month"},
            {"id": g + "bd", "name": f"{name} believed delay",
             "eq": str(delay), "units": "month"},
            {"id": g + "n", "name": f"{name} needed", "units": "WTE",
             "eq": f"Desired_treatments / {k}_productivity"}]
        caps.append(f"{k} * {k}_productivity")
    cap = caps[0] if len(caps) == 1 else "MIN(" + ", ".join(caps) + ")"
    vars_.append({"id": "cap", "name": "Capacity", "eq": cap,
                  "units": "patients/month"})
    return {"name": "Waiting list with hiring", "timeUnit": "month",
            "T": 120, "dt": 0.125, "stocks": stocks, "flows": flows,
            "vars": vars_}


case = lb.Model.from_spec(hiring_spec([("Consultants", 79, 50, 12)]))
print(case.equations())

Check the units as you read: recruiting is in WTE a month, the staff gap is in WTE divided by months, and `Consultants needed` is patients a month divided by patients per WTE per month, which is WTE. Now run the rule as it is usually applied, ignoring the pipeline (`pipeline_weight=0`), against the same rule counting it (`pipeline_weight=1`).

In [ ]:
#| fig-alt: "Two panels over 120 months. Left, the waiting list for the rule that ignores the pipeline swings from about 34,000 down to about 15,000 and back up, while the rule that counts it peaks near 35,000 and settles close to the target of 32,000. Right, consultants: ignoring the pipeline overshoots to about 100 and later 114 against a need of 88; counting it peaks near 94 and settles at 88."
def month_axis(ax):
    """Label a month axis with the running case's calendar years."""
    ticks = np.arange(0, 121, 24)
    ax.set_xticks(ticks, [str(2014 + t // 12) for t in ticks])


as_is = case.run(pipeline_weight=0, warn_clips=False)
fixed = case.run(pipeline_weight=1, warn_clips=False)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
for label, d in (("ignores the pipeline", as_is),
                 ("counts the pipeline", fixed)):
    d["Waiting list"].plot(ax=a1, label=label)
    d["Consultants"].plot(ax=a2, label=label)
a1.axhline(32000, color="grey", lw=1, ls=":")
a2.axhline(4400 / 50, color="grey", lw=1, ls=":")
a1.set(title="Waiting list", xlabel="year", ylabel="patients")
a2.set(title="Consultants", xlabel="year", ylabel="consultants (WTE)")
for a in (a1, a2):
    month_axis(a)
a1.legend(fontsize=8)
plt.tight_layout()
plt.show()

for label, d in (("ignores", as_is), ("counts", fixed)):
    print(f"{label:<8} list peaks", [(round(t), round(v))
                                     for t, v in peaks(d["Waiting list"])],
          "| consultant peaks", [(round(t), round(v, 1))
                                 for t, v in peaks(d["Consultants"])])
rec = as_is["Consultants recruiting"]
stopped = rec[(rec.index > 12) & (rec == 0)].index
end = stopped[stopped < 100].max()
print(f"ignoring the pipeline, no consultant is recruited from month "
      f"{stopped.min():.1f} to month {end:.1f}")

Figure: The waiting-list model with hiring, January 2014 to December 2023, with referrals 10% higher from January 2015. Left, the list against its target of 32,000 patients; right, consultants against the 88 WTE needed to treat 4,400 patients a month.

The rule that ignores the pipeline sees the list grow to 34,018 by month 20 and keeps adding posts, because the consultants it has already recruited have not started. By month 35 there are 100.1 consultants, 12 more than needed, and the list falls far below target, to about 18,000 in 2019. Then the mirror image: with the list low, the desired number of consultants is low, and no consultant is recruited at all from month 24.4 to month 84.9, more than five years. Leavers are not replaced, the team shrinks to about 72 in 2021, the list climbs to 35,641 in month 99, and a second hiring wave overshoots further, to 114.4 consultants by the end of 2023. **The swing grows**: each wave is bigger than the last.

Counting the pipeline changes the picture. The list peaks later and higher, at 35,259 in month 24 against 34,018 in month 20, because the rule stops adding posts once enough are on their way; consultants peak at 93.6, and the system settles with a small, slow ripple (a second list peak of 33,101 in month 83). The ripple that remains comes from the moving target: the list is itself a stock, so a hiring rule driven by the list has a second delay in its loop even when it sees the whole pipeline.

### What does the fix cost?

A decision maker will ask what is lost by being more careful. The scorecard below measures each run on the things a board watches: how high and how low the list goes, how much it still swings after year 5, the largest number of consultants, how many are hired in total, the **patient-months above target** (the excess list added up over time, a measure of extra waiting) and the **idle consultant-months** (capacity with no patients to treat). The last column counts the pipeline but believes the hiring delay is 6 months when it is 12.

In [ ]:
def scorecard(d, target=32000, dt=0.125):
    """What the decision maker cares about, from one run."""
    lst, late = d["Waiting list"], d["Waiting list"][d.index >= 60]
    return pd.Series({
        "list peak": lst.max(),
        "list low": lst.min(),
        "swing after year 5": late.max() - late.min(),
        "consultant peak": d["Consultants"].max(),
        "consultants hired": d["Consultants recruiting"].sum() * dt,
        "patient-months above target":
            (lst - target).clip(lower=0).sum() * dt,
        "idle consultant-months":
            (d["Capacity"] - d["Treatments"]).sum() * dt / 50})


table = pd.DataFrame({
    "ignores, 10 years": scorecard(as_is),
    "counts, 10 years": scorecard(fixed),
    "ignores, 20 years": scorecard(case.run(T=240, warn_clips=False)),
    "counts, 20 years": scorecard(case.run(T=240, pipeline_weight=1)),
    "counts, believes 6": scorecard(case.run(
        pipeline_weight=1, consultants_believed_delay=6)),
})
print(table.round(0).astype(int).to_string())

Over the first ten years, ignoring the pipeline has one thing going for it: its over-hiring drives the list far below target, so it adds up fewer patient-months above target (58,575) than the careful rule (83,015). That is the **cost of the fix**: the careful rule hires more cautiously at first, so patients wait longer in years 2 to 4. Everything else favours it. It hires 96 consultants in ten years against 127, never needs more than 94 consultants against 114, and keeps the list within a band of 2,648 patients after year 5 against 20,361. Over twenty years even the waiting argument reverses: the swinging rule ends up with more patient-months above target (105,767 against 91,447), hires 230 consultants against 185, and leaves 491 consultant-months idle when the list is driven down to its floor.

The last column is the warning. Counting the pipeline is only as good as the estimate of the delay. A rule that believes recruitment takes 6 months keeps too few posts in the pipeline; it never fully catches up, and the list sits above target for good, adding up 193,552 patient-months above target, more than twice the well-informed rule. Measuring the real hiring delay is part of the fix.

### Adding nurses

Consultants are not the only staff. Nurses can usually be hired faster, in about 4 months in our assumption, and each nurse supports 25 patients a month. With capacity set by the scarcer group, `Capacity = MIN(Consultants * 50, Nurses * 25)`, the same function builds a two-group model.

In [ ]:
#| fig-alt: "Waiting list over 120 months for three versions of the hiring rule that ignores the pipeline: consultants only with a 12-month delay; consultants and nurses, with nurses hired in 4 months; and the same with the pipeline counted. Adding fast-hired nurses damps the swing in the list."
team = lb.Model.from_spec(hiring_spec([("Consultants", 79, 50, 12),
                                       ("Nurses", 158, 25, 4)]))
fig, ax = plt.subplots()
for label, d in (("consultants only, ignores", as_is),
                 ("with nurses, ignores", team.run(warn_clips=False)),
                 ("with nurses, counts", team.run(pipeline_weight=1))):
    d["Waiting list"].plot(ax=ax, label=label)
    extra = (f", nurses peak {d['Nurses'].max():.0f}"
             if "Nurses" in d else "")
    print(f"{label:<27} list {d['Waiting list'].min():6.0f} to "
          f"{d['Waiting list'].max():6.0f}, consultants peak "
          f"{d['Consultants'].max():5.1f}{extra}")
ax.axhline(32000, color="grey", lw=1, ls=":")
ax.set(title="Adding nurses who can be hired quickly",
       xlabel="year", ylabel="waiting list (patients)")
month_axis(ax)
ax.legend(fontsize=8)
plt.show()
d = team.run(warn_clips=False)
idle = (d["Consultants"] * 50 - d["Capacity"]).clip(lower=0) / 50
print(f"with nurses, ignores: consultants without a nurse team at the "
      f"peak: {idle.max():.1f} WTE in month {idle.idxmax():.0f}")

Figure: The waiting list when the hiring rule ignores the pipeline, for consultants alone and with nurses added (hired in 4 months), and with nurses when the pipeline is counted.

Adding fast-hired nurses damps the swing in the list a lot: from 15,280 to 35,641 with consultants alone, to 28,953 to 34,018 with nurses. The nurse loop is quick, so capacity cannot run far ahead of what is needed: when consultants overshoot, nurses become the limit. But the consultant swing has not gone away; it has moved out of the list and into idle posts. Consultants still overshoot to 100.1, and at the worst point, in month 50, 13.3 consultant WTE have no nurse team to work with. A model that aggregates staff into one number would miss that cost. Counting the pipeline still helps: consultants peak at 93.6 instead of 100.1.

**Try it 5.**

- *Check.* At the start of the run, how many consultants leave each month, and how many are in the pipeline? Show that the second is the first times the hiring delay.
- *Apply.* With the pipeline still ignored, shorten the hiring delay to 6 and to 3 months. How big is the swing in the list now?
- *Extend.* With the pipeline counted, try a time to reduce the list of 6, 12 and 24 months. What do you gain and lose in swing and in waiting?

<details><summary>Worked answer</summary>

0.658 consultants leave each month (79 / 120), and the pipeline holds 7.9 WTE, which is 0.658 times 12 months. Shortening the hiring delay to 6 months narrows the list to between 27,456 and 33,542, and to 3 months between 30,739 and 33,281; consultants peak at 95.8 and 93.4. Faster recruitment is as good a cure as counting the pipeline, but it is usually much harder to buy. A gentler list target trades waiting for calm: at 12 months the late swing falls from 2,648 to 1,494 patients but the patient-months above target rise from 83,015 to 111,933; at 24 months the waiting cost climbs to 180,380 and the swing is no smaller (2,196). There is no free setting; the analyst's job is to put the trade-off in front of the decision maker.
</details>

In [ ]:
# Check: the desired pipeline at the start
d0 = case.run(T=0)
print("consultants leaving", round(d0["Consultants leaving"].iloc[0], 3),
      "WTE/month; pipeline", round(d0["Consultants in pipeline"].iloc[0], 2),
      "WTE")
# Apply: a shorter hiring delay, pipeline still ignored
for delay in (12, 6, 3):
    d = case.run(consultants_hiring_delay=delay,
                 consultants_believed_delay=delay, warn_clips=False)
    print(f"hiring delay {delay:>2} months: list "
          f"{d['Waiting list'].min():6.0f}"
          f" to {d['Waiting list'].max():6.0f}, consultants peak "
          f"{d['Consultants'].max():5.1f}")
# Extend: a gentler list target (12 months) with the pipeline counted
for tr in (6, 12, 24):
    s = scorecard(case.run(pipeline_weight=1, time_to_reduce_list=tr))
    print(f"time to reduce list {tr:>2}: swing after year 5 "
          f"{s['swing after year 5']:5.0f}, patient-months above target "
          f"{s['patient-months above target']:7.0f}")

## 6. Swings in markets: when price is the signal

In a market nobody manages the whole stock. Each producer reacts to a price, and the price carries the gap between demand and supply. How the price moves therefore matters as much as the delays in building capacity. Three molecules give three rules.

**Smooth Pricing** (`smoothPricing`), after Hines, *Molecules of Structure* [@hines_molecules], lets price ease towards an indicated price set by a reference price and market pressure, over a price adjustment time $T_p$: $\; dP/dt = (P_{\text{ref}} \times \text{pressure} - P)/T_p$. It settles once it gets there.

**Sea Anchor Pricing** (`seaAnchorPricing`), after Hines, *Molecules of Structure* [@hines_molecules], uses the anchoring rule of chapter 5. Price setters mark an underlying price $U$ up or down for pressure from inventory and market share, and the underlying price drifts towards the price actually charged:

$$
P = U \times \frac{2}{1 + I/I^{*}} \times \frac{2\,m}{1 + m}, \qquad \frac{dU}{dt} = \frac{P - U}{T_U},
$$

with inventory $I$ against its target $I^{*}$ and relative market share $m$. While the pressure lasts, the price keeps moving. **Protected Sea Anchor Pricing** (`protSeaAnchorPricing`), after Hines, *Molecules of Structure* [@hines_molecules], adds a floor: the underlying price never chases the charged price below a minimum regarded as sustainable. In these molecules inventory is held fixed, so we see the pricing rule on its own; in a full market model the price would feed back to supply and demand.

In [ ]:
#| fig-alt: Price over 30 years for four pricing rules. Smooth Pricing rises from 10 and levels off at 14. Sea Anchor Pricing with inventory short keeps climbing. With inventory in glut, Sea Anchor Pricing falls towards zero, while Protected Sea Anchor Pricing levels off near its floor.
sp = lb.molecule("smoothPricing")
sa = lb.molecule("seaAnchorPricing")
ps = lb.molecule("protSeaAnchorPricing")
runs = {
    "Smooth, demand up 40%": sp.run(T=30, market_pressure=1.4)["Price"],
    "Sea anchor, inventory 75": sa.run(T=30, inventory=75)["Price"],
    "Sea anchor, inventory 150": sa.run(T=30, inventory=150)["Price"],
    "Protected, inventory 150":
        ps.run(T=30, preset="Glut with $6 floor")["Price"],
}
fig, ax = plt.subplots()
for label, p in runs.items():
    p.plot(ax=ax, label=label)
ax.set(title="Four ways a price responds to pressure", xlabel="time (year)",
       ylabel="price (money/unit)")
ax.legend(fontsize=8)
plt.show()
print(pd.DataFrame(runs).loc[[0.0, 4.0, 10.0, 20.0, 30.0]].round(2)
      .to_string())

Figure: Four pricing rules over 30 years. Smooth Pricing with 40% more market pressure; Sea Anchor Pricing with inventory at 75% and at 150% of target; Protected Sea Anchor Pricing in the same glut with a floor of 6 on the underlying price.

The smooth price goes from 10 to 12.55 after one adjustment time of 4 years and settles at 14.00. The anchored price never settles: with inventory short, it climbs from 11.43 to 33.30 in 30 years, because every year's mark-up is applied to an anchor that has itself risen. In a glut the same rule drives the price down without end, to 1.78. The protected version stops the slide: the underlying price holds at its floor and the price charged settles at 4.81. In a real market, a rising price would bring new capacity and cut demand; because capacity takes years to build, that response arrives late and goes too far, and the anchored price turns a delay into a cycle.

### Swinging systems in the real world

The same structure, a stock corrected through a delay that decisions underweight, turns up across sectors and countries. Three short cases, told in our own words:

- **Housing in Ireland.** House completions rose through the early 2000s to a peak of roughly 90,000 a year in 2006, as builders started projects on the strength of rising prices. Homes take years from site purchase to sale, and the starts already in progress were not counted against future demand. When the market turned, completions collapsed to under 10,000 a year by 2013, and the country moved from a glut of empty estates to a shortage within a decade. The stock is homes, the pipeline is houses under construction, and prices carry the signal.
- **Pork in China and Germany.** Farmers expand breeding herds when pork prices are high; the extra pigs reach the market about a year or more later, prices fall, and herds are cut. German economists described this *Schweinezyklus* (pig cycle) in the 1920s. China, the largest pork market, saw a sharp version after African swine fever cut the national herd in 2018 and 2019: prices roughly doubled, farmers rebuilt herds quickly, and by 2021 prices had fallen steeply on oversupply.
- **Nurses in the Philippines.** Strong overseas demand for nurses in the 2000s drew large numbers of students into nursing schools, a pipeline of four or more years. By the time they graduated, overseas hiring had slowed, and around 2010 many newly licensed nurses could not find nursing work. The stock is trained nurses, the pipeline is students, and the signal is the job market several years earlier.

| Case | Stock | Supply line | Delay | Signal that drives the decision |
|---|---|---|---|---|
| Hospital hiring (our case) | consultants | posts being recruited | about 12 months | waiting list above target |
| Irish housing | homes | homes under construction | 2 to 3 years | house prices |
| Pig cycle | breeding herd | animals being raised | about 1 to 2 years | pork prices |
| Philippine nurses | licensed nurses | students in training | 4 years or more | job market and overseas demand |

In each case the decision makers are reacting sensibly to what they can see. What they cannot see, or do not count, is what is already on its way.

**Try it 6.**

- *Check.* With inventory at 75 of a target of 100 and a market share of 1, what is the pressure to change price in Sea Anchor Pricing?
- *Apply.* How long does the sea-anchored price take to double with inventory at 75? Predict it from the pressure first.
- *Extend.* In the protected glut, where do the underlying price and the price charged settle? Explain the gap between them.

<details><summary>Worked answer</summary>

The pressure is 2 / (1 + 0.75) = 1.1429. The price charged is 14.3% above the anchor, which closes that gap over 4 years, so the anchor grows at about 14.3% / 4 = 3.6% a year and doubles in about 0.69 / 0.036, or 19.4 years. The run says year 19.5. In the protected glut the underlying price settles at its floor of 6.00 while the price charged is 4.80: the pressure, 2 / (1 + 1.5) = 0.8, still marks the price down from the anchor, but the anchor no longer follows. A floor on the anchor, not on the price, is what stops the spiral.
</details>

In [ ]:
# Check: the pressure on price when inventory is 75 of a target of 100
d = sa.run(T=0, inventory=75)
print("pressure", round(d["Pressure to change price"].iloc[0], 4))
# Apply: doubling time of the sea-anchored price
p = sa.run(T=60, inventory=75)["Price"]
print("price doubles by year", p.index[p >= 2 * p.iloc[0]][0])
# Extend: where the protected price settles in a glut
d = ps.run(T=60, preset="Glut with $6 floor")
print("protected: underlying", round(d["Underlying price"].iloc[-1], 2),
      "price", round(d["Price"].iloc[-1], 2),
      "| pressure", round(d["Pressure to change price"].iloc[-1], 4))

## Check it

Three habits for any stock management model: every element has units; a model started in balance with no shock does not move; and halving the time step does not change the story.

In [ ]:
# 1. Units: every element of the case model has one
print("missing units:", [e["name"] for e in case.stocks + case.flows
                         + case.vars if not e.get("units")])
# 2. Extreme conditions: a balanced start with no shock must not move
calm = case.with_init("Consultants", 80).run(referral_rise=0,
                                             warn_clips=False)
print("balanced start: list from", calm["Waiting list"].min(), "to",
      calm["Waiting list"].max())
# 3. Halve dt: the swing should not be an artefact of the time step
for dt in (0.25, 0.125, 0.0625):
    d = case.run(dt=dt, warn_clips=False)
    print(f"dt {dt:<6}: list in month 120 {d['Waiting list'].iloc[-1]:8.1f},"
          f" consultant peak {d['Consultants'].max():6.2f}")

Every element has units. Started with 80 consultants (exactly enough for 4,000 referrals a month) and no rise in referrals, the list stays at 32,000 patients throughout: the model is in equilibrium where it should be. Halving `dt` from 0.25 to 0.125 to 0.0625 month moves the list in month 120 from 16,197 to 15,280 to 14,937 patients, and the consultant peak from 116.4 to 114.4 to 113.4. The changes shrink by about two thirds with each halving, so the numbers are converging, and the swing is a property of the model, not of the step. Because the swinging run is near the edge of stability, though, its late values are sensitive; quote them to two significant figures, not five.

## Running case

This chapter added staff to the waiting list:

| Element | What it is | Value |
|---|---|---|
| `Consultants` (stock) | consultant WTE, each with a team | starts at 79; 50 patients a month each |
| `Consultants in pipeline` (stock) | posts recruited but not yet started | starts at 7.9 WTE |
| `Consultants hiring delay` | time from recruiting to starting | 12 months |
| `Average tenure` | time a consultant stays | 120 months |
| `Desired treatments` | referrals plus the excess list over `Time_to_reduce_list` | 6 months |
| `Pipeline_weight` | share of the pipeline gap the rule counts | 0 as found, 1 as fixed |
| `Nurses` (optional group) | nurse WTE, hired in 4 months | 158, 25 patients a month each |

Chapter 7, Patterns that repeat, adds the quick fixes managers use while the slow hiring loop catches up (overtime, rushed clinics and outsourcing) and shows how each can backfire.

## Tell the decision maker

- **Hiring to the list will overshoot unless posts already in the pipeline are counted.** In our run, a rule that ignores them hires 127 consultants in ten years where 96 would do, peaks at 114 consultants against a need of 88, and stops recruiting entirely for more than five years in between.
- **Counting the pipeline costs some waiting early on.** The careful rule adds about 24,000 more patient-months above target in the first ten years, but over twenty years it does better on waiting as well as on staff.
- **The fix needs data**: a single register of posts approved, advertised, offered and accepted, with realistic start dates. A rule that underestimates the hiring delay by half leaves the list above target for good.
- **Faster recruitment is the other cure.** Cutting the hiring delay from 12 to 3 months removes most of the swing even with the old rule.

## Going further

- The period and damping of a stock management loop can be found without simulating, by linearising the model and computing the eigenvalues of its Jacobian; `numpy.linalg.eigvals` does this in one line once the matrix is written down. Complex eigenvalues mean oscillation; a positive real part means the swing grows, as in our case.
- Experimental work with supply chain games shows that people typically give the supply line a weight well below 1, which is why oscillation is so common in practice [@sterman2000]. @forrester1961 first showed how such ordering rules amplify swings along a production and distribution chain.
- Business cycles arise when inventory and workforce are both managed this way inside many firms at once; long construction delays give the slower cycles of housing and commodities.

## Mini-project

Take the two-group model (consultants and nurses) and drive it with a shock like the one in the data in 2020: from month 74, capacity falls to 35% and recovers linearly over 9 months (hint: multiply `Capacity` by a `TABLE` of time with `with_equation`). Compare the rule that ignores the pipeline with the one that counts it, using the scorecard. Then write a half-page note to the board: how many consultants and nurses should be recruited in the year after the shock, and what should they expect the list to do over the following five years?

## Further reading

- @sterman2000 treats stock management, supply chains and business cycles at length; read it for the experimental evidence and the classic supply chain models.
- @forrester1961, *Industrial Dynamics*, is where the amplification of swings along supply chains was first modelled.
- @meadows2008 gives a short, non-technical account of delays and oscillation in everyday systems.
- Hines, *Molecules of Structure* [@hines_molecules], describes the stock management, workforce and pricing structures used here.

If you would like to play with the molecule behind this chapter, the LoopBricks app has an interactive version: [Stock Management](https://waelrash1.github.io/loopbricks/#/m/stockmgmt).